# Buổi 9 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `dac_trung.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Chạy code đầu buổi (khoảng 1 phút)

In [ ]:
%matplotlib inline
import warnings

import dac_trung as dt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
tat_ca = dt.doc_tsf()
chuoi = dt.lay_mau(tat_ca, 4000)
bang = dt.bang_dac_trung(chuoi).join(dt.danh_gia_kho_de(chuoi))
print(len(tat_ca), "chuỗi M4;", len(chuoi), "chuỗi mẫu, mỗi chuỗi", len(next(iter(chuoi.values()))), "tháng")
print(dt.tuong_quan_kho_de(bang).round(3).to_string(index=False))
print(dt.de_xuat_chien_luoc(bang).value_counts().to_dict())

mau300 = {t: chuoi[t] for t in list(chuoi)[:300]}
nhan, _ = dt.phan_cum_dtw(mau300)
muc = pd.Series({t: np.mean(v) for t, v in mau300.items()})
print("mức trung vị theo cụm DTW:", muc.groupby(nhan).median().round(0).to_dict())

## Bước 2 — Đặc trưng và entropy trên chuỗi nhỏ (mục 4.1, 4.2)

In [ ]:
t = np.arange(120)
y = 50 + 0.2 * t + 5 * np.sin(2 * np.pi * t / 12) + np.random.default_rng(3).normal(0, 2, 120)
goc, lon = dt.dac_trung_mot_chuoi(y), dt.dac_trung_mot_chuoi(1000 * y)
print(pd.DataFrame({"chuỗi gốc": goc, "nhân 1.000": lon, "tỷ số": {k: lon[k] / goc[k] if goc[k] else np.nan for k in goc}}).round(3))

rng = np.random.default_rng(0)
mua_vu = np.sin(2 * np.pi * t / 12) + 0.2 * rng.normal(size=120)
nhieu = rng.normal(size=120)
print("entropy: mùa vụ", round(dt.entropy_pho(mua_vu), 2), "| nhiễu thuần", round(dt.entropy_pho(nhieu), 2))

## Bước 3 — Entropy so với sai số thật (mục 4.3)

Sửa `tuong_quan_kho_de` rồi chạy lại ô này.

In [ ]:
tq = dt.tuong_quan_kho_de(bang)
print(tq.round(3).to_string(index=False))
nhom = pd.qcut(bang["entropy_pho"], 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"])
cot = [c for c in ("smape_snaive", "mase_snaive", "mase_naive1_snaive") if c in bang]
print(bang.groupby(nhom, observed=True)[cot].median().round(2))

con = tq[tq["đặc trưng"] == "entropy_pho"]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(np.arange(len(con)) - 0.2, con["pearson"], 0.4, label="Pearson")
if "spearman" in con:
    ax.bar(np.arange(len(con)) + 0.2, con["spearman"], 0.4, label="Spearman")
ax.set_xticks(np.arange(len(con)), con["thước đo"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("tương quan với entropy")
ax.legend();

## Bước 4 — Bản đồ và chiến lược (mục 4.4)

Sửa `de_xuat_chien_luoc` rồi chạy lại ô này.

In [ ]:
toa_do, pca = dt.khong_gian_dac_trung(bang)
print("PC1, PC2 giữ", pca.explained_variance_ratio_.round(3))
fig, truc = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, c in zip(truc, ["entropy_pho", "do_manh_mua_vu", "smape_snaive"], strict=True):
    h = ax.scatter(toa_do[:, 0], toa_do[:, 1], c=np.clip(bang[c].to_numpy(float), 0, 40), s=4)
    plt.colorbar(h, ax=ax)
    ax.set_title(c)
    ax.set_xlabel("PC1")
plt.show()

chien_luoc = dt.de_xuat_chien_luoc(bang)
print(chien_luoc.value_counts().to_dict())
print(bang.groupby(chien_luoc)["smape_snaive"].median().round(2).to_dict())

xa = pd.Series(np.hypot(toa_do[:, 0], toa_do[:, 1]), index=bang.index).nlargest(3).index
fig, truc = plt.subplots(1, 3, figsize=(12, 2.8))
for ax, ten in zip(truc, xa, strict=True):
    ax.plot(chuoi[ten])
    ax.set_title(ten)
plt.show()
print(bang.loc[xa, ["doan_phang", "kpss_p", "spike", "ty_le_0", "entropy_pho"]].round(3))

## Bước 5 — Phân cụm hình dạng và ABC–XYZ (mục 4.5, 4.6)

Sửa mặc định `chuan_hoa` trong `phan_cum_dtw` rồi chạy lại ô này. Cuối cùng: `python lab.py check` trong terminal.

In [ ]:
nhan, _ = dt.phan_cum_dtw(mau300)
print("mức trung vị theo cụm DTW:", muc.groupby(nhan).median().round(0).to_dict())

thu = dict(list(mau300.items())[:40])
dau = next(iter(thu))
thu["nhân 100"] = thu[dau] * 100
nhan_thu, _ = dt.phan_cum_dtw(thu)
print("nhân một chuỗi với 100 vẫn cùng cụm:", nhan_thu["nhân 100"] == nhan_thu[dau])

abc = dt.abc_xyz(dt.doc_ban_le())
print(len(abc), "mã hàng; A chiếm", round(abc.loc[abc["abc"] == "A", "sum"].sum() / abc["sum"].sum(), 3), "doanh thu")
print(abc.groupby(["abc", "xyz"]).size().unstack(fill_value=0))